# Binay Karki | MAP003 V04 — Ward 16 satellite coverage recovery

Run all once and approve authentication. This extraction-only notebook exports a new, separately named GeoTIFF on the original 30 m grid. It preserves the V03 strict raster and all prior exports. The original 11 scene product IDs and dates are fixed. No plotting or temperature interpolation occurs here.

Strict LST remains primary. The new basic and moderate quality layers are independent sensitivity data, not a replacement for strict observations. ChatGPT Python will use the original Ward 6 and 8 maps unchanged; only Ward 16's missing strict cells will be considered for supplementary display after raster QA. NoData stays -9999; zero in counts/flags is not a temperature. Manual observations are not used to fill satellite gaps.

Permanent Colab ID: `1Jf2JZwiIodAHMXn7Xr2XYvOifqnJ8FXv`; local archive event `000004` (not a global A7 sequence).


## 1. Configuration — editable settings and stable source identities

In [ ]:
from pathlib import Path
import os, io, json, csv, hashlib, time, sys, subprocess, datetime

STABLE_COLAB_DRIVE_ID = "1Jf2JZwiIodAHMXn7Xr2XYvOifqnJ8FXv"
PRE_ARCHIVE_DRIVE_ID = "15uAQVW6gtREzRh9lh7H1jvBjMkmL6bwI"  # V04 PRE: executed V03 with outputs
EE_PROJECT_ID = "just-a-project-5007"   # This is the project ID in Binay's archived executed notebook. Change only if access has changed.
START_DATE = "2026-05-31"
END_EXCLUSIVE = "2026-08-13"  # Includes August 12, matching the historic GEE-V05 source period.
GRID_CRS, GRID_SCALE_METRES = "EPSG:32644", 30
SOURCE_MUNICIPAL_DRIVE_ID = "1PxGcC1OMT8A7AZpa_a2HCUgKzURTyT7l"
SOURCE_MUNICIPAL_SHA256 = "b7ed29c435f736d436150ce735692990b320bef1e64b9a77d046fd39736d19f5"
SOURCE_MUNICIPAL_GIT_URL = "https://raw.githubusercontent.com/FabinGurung/jp-research-lab/main/researchers/binay-karki/gis/map002/pokhara_metropolitan_historical_RMP_NepalDD_to_WGS84.geojson"
OUTPUT_FOLDER_ID = "1PH3JhBcsSnDMxAx4B4-9y_50ZLEbm4fd"
OUTPUT_FOLDER_NAME = "BINAY_MAP003_CITYWIDE_WET_2026_EXPORTS"  # Earth Engine uses folder NAME, never ID.
RUN_KEY = "BINAY_MAP003_CITYWIDE_WET_LST_2026_V04_COVERAGE_QA"
NODATA = -9999.0
RUN_EXPORT = True  # Requires successful task-list OAuth preflight in Section 3.
MAP003_WORKING_VERSION = "V04"
EE_AUTH_MODE = "colab"  # If scopes are insufficient: change to "gcloud" and rerun the entire notebook.
MAP003_ARCHIVE_EVENT = "000004"  # Local PRE/POST ordinal, NOT global A7 SEQ.
OUTPUT_DIR = Path('/content/binay_map003_citywide')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('MAP003 V04 coverage recovery extraction-only. Run all submits/reuses a task; it does not wait for completion.')
# Exact alignment to the original completed V01 GeoTIFF, read with rasterio.
GRID_TRANSFORM = [30, 0, 774510, 0, -30, 3139020]
GRID_BOUNDS = [774510, 3108930, 809580, 3139020]
ORIGINAL_STRICT_RASTER_DRIVE_ID = "1OpMn-3fsSQtYQQXmWbN4sDLnkRQ_b4Js"
ORIGINAL_STRICT_RASTER_SHA256 = "2a6a9440d834aaf55e3de72b9c79bb34c807c26850e605b6f0e57b5523bc53d7"


## 2. Install extraction dependencies and SHA-check the municipal polygon

In [ ]:
# No rasterio, matplotlib, plotting packages, or Google Sheets conversion here.
required = ['earthengine-api','geopandas','pyproj','pandas','requests',
            'google-api-python-client','google-auth']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *required])
import requests, geopandas as gpd, ee, pandas as pd
from IPython.display import display

raw = requests.get(SOURCE_MUNICIPAL_GIT_URL, timeout=60)
raw.raise_for_status()
raw_bytes = raw.content
actual_source_sha = hashlib.sha256(raw_bytes).hexdigest()
assert actual_source_sha == SOURCE_MUNICIPAL_SHA256, (
    f'Municipal polygon SHA mismatch: {actual_source_sha}. STOP: no GEE export.')
municipality_file = OUTPUT_DIR / 'pokhara_metropolitan_historical_RMP_NepalDD_to_WGS84.geojson'
municipality_file.write_bytes(raw_bytes)
municipality_geojson = json.loads(raw_bytes)
assert municipality_geojson['type']=='FeatureCollection' and len(municipality_geojson['features'])==1
municipality_geom = municipality_geojson['features'][0]['geometry']
assert municipality_geom['type'] in ('Polygon','MultiPolygon')
local_mun = gpd.read_file(municipality_file).to_crs(GRID_CRS)
assert local_mun.geometry.is_valid.all()
municipal_planar_km2 = float(local_mun.geometry.area.sum()/1e6)
assert 460.0 < municipal_planar_km2 < 470.0, f'Unexpected municipal polygon area: {municipal_planar_km2}'
print('Historic municipal polygon SHA VERIFIED:', actual_source_sha)
print('UTM 44N planar area:', round(municipal_planar_km2,4), 'km²; not official 2026 certification')

## 3. Authenticate Earth Engine and preflight batch-task permissions

Approve the Google authentication dialog if prompted. The saved V01 run failed on
`ee.batch.Task.list()` with insufficient scopes; V02's explicit preflight is
preserved. If necessary, set `EE_AUTH_MODE = "gcloud"` in Section 1, then rerun.

In [ ]:
# Required before any Earth Engine export or task-status operations.
# A normal successful EE image.getInfo() is NOT proof of task-management scopes.
AUTH_EE_READY = False
EE_TASK_SNAPSHOT = None

def check_earth_engine_batch_access():
    # The same API that failed in saved V01 execution (via listOperations).
    tasks = ee.batch.Task.list()
    if not isinstance(tasks, list):
        raise RuntimeError('Unexpected EE task-list response; stop before export.')
    return tasks

try:
    ee.Authenticate(force=True, auth_mode=EE_AUTH_MODE)  # Fresh user OAuth consent
    ee.Initialize(project=EE_PROJECT_ID)
    EE_TASK_SNAPSHOT = check_earth_engine_batch_access()
except Exception as exc:
    AUTH_EE_READY = False
    if any(term in str(exc).lower() for term in
           ('insufficient authentication scopes','insufficient permissions','missing required scopes')):
        raise RuntimeError(
            'Earth Engine operations still lack OAuth scopes. NO export was launched. '
            'Change EE_AUTH_MODE to "gcloud" in Section 1, then rerun Section 3; '
            'this forces a new signed-in OAuth flow without reverting to Colab mode. '
            'Rerun this section (or run check_earth_engine_batch_access() directly '
            'after reauthorization). Authorize the required Earth Engine, '
            'Cloud and Google Drive permissions in the signed-in browser. '
            'See https://developers.google.com/earth-engine/guides/auth'
        ) from exc
    raise
else:
    AUTH_EE_READY = True
    city = ee.Geometry(municipality_geom, proj='EPSG:4326', geodesic=False)
    print('EE TASK LIST OAUTH PREFLIGHT PASSED:', EE_PROJECT_ID)
    print('Tasks visible:', len(EE_TASK_SNAPSHOT))
    print('Historical Pokhara municipal AOI ready. No new export started.')


## 4. Discover and log original Landsat 8/9 scene IDs

In [ ]:
def collection(dataset_id):
    return (ee.ImageCollection(dataset_id)
      .filterBounds(city)
      .filterDate(START_DATE, END_EXCLUSIVE)
      .filter(ee.Filter.eq('PROCESSING_LEVEL','L2SP')))

l8 = collection('LANDSAT/LC08/C02/T1_L2')
l9 = collection('LANDSAT/LC09/C02/T1_L2')
FROZEN_PRODUCT_IDS = ['LC08_L2SP_142040_20260606_20260613_02_T1', 'LC08_L2SP_142041_20260606_20260613_02_T1', 'LC09_L2SP_142040_20260614_20260616_02_T1', 'LC09_L2SP_142041_20260614_20260616_02_T1', 'LC08_L2SP_142040_20260622_20260627_02_T1', 'LC08_L2SP_142041_20260622_20260627_02_T1', 'LC09_L2SP_142040_20260630_20260701_02_T1', 'LC08_L2SP_142040_20260708_20260716_02_T1', 'LC09_L2SP_142040_20260801_20260802_02_T1', 'LC09_L2SP_142041_20260801_20260802_02_T1', 'LC08_L2SP_142040_20260809_20260816_02_T1']
raw_scenes = l8.merge(l9).filter(ee.Filter.inList('LANDSAT_PRODUCT_ID', FROZEN_PRODUCT_IDS)).sort('system:time_start')
actual_ids = raw_scenes.aggregate_array('LANDSAT_PRODUCT_ID').getInfo()
if sorted(actual_ids) != sorted(FROZEN_PRODUCT_IDS):
    raise RuntimeError('Original scene set changed or unavailable; stop before export, do not silently substitute.')
scene_count = int(raw_scenes.size().getInfo())
if scene_count == 0:
    raise RuntimeError('No L2SP scenes intersect municipal AOI in this period: STOP (no synthetic temperatures).')

props = ['LANDSAT_PRODUCT_ID','SPACECRAFT_ID','system:time_start','WRS_PATH','WRS_ROW','CLOUD_COVER']
arrays = {k:raw_scenes.aggregate_array(k).getInfo() for k in props}
scene_records=[]
for idx in range(scene_count):
    r={k:arrays[k][idx] if idx<len(arrays[k]) else None for k in props}
    ts=r.get('system:time_start')
    r['acquired_UTC']=datetime.datetime.fromtimestamp(ts/1000, datetime.timezone.utc).isoformat() if ts else None
    scene_records.append(r)
scene_df=pd.DataFrame(scene_records).sort_values(['acquired_UTC','LANDSAT_PRODUCT_ID'])
scene_csv=OUTPUT_DIR / f'{RUN_KEY}_EXACT_LANDSAT_SCENES.csv'
scene_df.to_csv(scene_csv, index=False)
print('Real 2026 historical municipal AOI scene counts: L8',raw_scenes.filter(ee.Filter.eq('SPACECRAFT_ID','LANDSAT_8')).size().getInfo(), 'L9',raw_scenes.filter(ee.Filter.eq('SPACECRAFT_ID','LANDSAT_9')).size().getInfo(),'combined',scene_count)
display(scene_df.head(15))
print('Full scene manifest CSV:',scene_csv)

## 5. Strict LST plus separately labelled sensitivity layers

Original V05 thresholds unchanged. Moderate sensitivity: ST_QA ≤ 7.5 K, cloud distance ≥ 0.5 km, 15–60°C. Basic-range sensitivity uses base QA and the same 15–60°C range. Raw basic temperatures are exported for diagnosis; they are not automatically accepted.


In [ ]:
# The archived V05 already applied these thresholds: reject QA_PIXEL bits 0–5;
# QA_RADSAT == 0; ST_QA <= 5 Kelvin; ST_CDIST >= 1 km; LST 15–60°C.
# Reusing QA gates avoids silently changing Binay's 2026 satellite scientific criteria.
V05_MAX_ST_QA_K = 5.0
V05_MIN_ST_CDIST_KM = 1.0
V05_LST_MIN_C, V05_LST_MAX_C = 15.0, 60.0

MODERATE_MAX_ST_QA_K = 7.5
MODERATE_MIN_ST_CDIST_KM = 0.5

def prep_citywide(image):
    qa = image.select('QA_PIXEL')
    bad = qa.bitwiseAnd(63).neq(0)   # bits 0,1,2,3,4,5 = fill, dilated cloud, cirrus, cloud, cloud shadow, snow
    not_saturated = image.select('QA_RADSAT').eq(0)
    lst_c = image.select('ST_B10').multiply(0.00341802).add(149).subtract(273.15)
    st_qa_k = image.select('ST_QA').multiply(0.01)
    cloud_distance_km = image.select('ST_CDIST').multiply(0.01)
    base_mask = bad.Not().And(not_saturated)
    primary_mask = (base_mask.And(st_qa_k.lte(V05_MAX_ST_QA_K))
                    .And(cloud_distance_km.gte(V05_MIN_ST_CDIST_KM))
                    .And(lst_c.gte(V05_LST_MIN_C)).And(lst_c.lte(V05_LST_MAX_C)))
    range_mask = lst_c.gte(V05_LST_MIN_C).And(lst_c.lte(V05_LST_MAX_C))
    basic_range = base_mask.And(range_mask)
    moderate = basic_range.And(st_qa_k.lte(MODERATE_MAX_ST_QA_K)).And(cloud_distance_km.gte(MODERATE_MIN_ST_CDIST_KM))
    # Mask diagnostic bands to pixels with actual basic thermal observations.
    observed_base = base_mask.And(lst_c.mask())
    return ee.Image.cat([
        lst_c.updateMask(base_mask).rename('LST_BASE_C'),
        lst_c.updateMask(primary_mask).rename('LST_PRIMARY_C'),
        lst_c.updateMask(moderate).rename('LST_MODERATE_C'),
        lst_c.updateMask(basic_range).rename('LST_BASIC_RANGE_C'),
        st_qa_k.updateMask(observed_base).rename('BASE_ST_QA_K'),
        cloud_distance_km.updateMask(observed_base).rename('BASE_CDIST_KM'),
    ]).copyProperties(image,['system:time_start','LANDSAT_PRODUCT_ID','SPACECRAFT_ID'])

processed = raw_scenes.map(prep_citywide)
city_median_lst = processed.select('LST_PRIMARY_C').median().rename('LST_C').clip(city)
n_primary = processed.select('LST_PRIMARY_C').count().rename('N_PRIMARY').unmask(0).clip(city)
n_base = processed.select('LST_BASE_C').count().rename('N_BASE').unmask(0).clip(city)
# A scene's geographic footprint is not proof of usable thermal measurements.
footprints = raw_scenes.map(lambda im: ee.Image.constant(1).rename('FP').clip(im.geometry()))
scene_footprint = footprints.max().unmask(0).rename('SCENE_FOOTPRINT').clip(city)
print('Prepared server-side citywide median with real masks and per-pixel observations; NOT yet exported.')
base_median = processed.select('LST_BASE_C').median().rename('LST_BASE_C').clip(city)
moderate_median = processed.select('LST_MODERATE_C').median().rename('LST_MODERATE_C').clip(city)
base_range_median = processed.select('LST_BASIC_RANGE_C').median().rename('LST_BASIC_RANGE_C').clip(city)
n_moderate = processed.select('LST_MODERATE_C').count().unmask(0).rename('N_MODERATE').clip(city)
n_basic_range = processed.select('LST_BASIC_RANGE_C').count().unmask(0).rename('N_BASIC_RANGE').clip(city)
base_qa_median = processed.select('BASE_ST_QA_K').median().rename('BASE_ST_QA_MED_K').clip(city)
base_cdist_median = processed.select('BASE_CDIST_KM').median().rename('BASE_CDIST_MED_KM').clip(city)
# Per-cell class: 3 strict; 2 moderate only; 1 basic range only; 0 no accepted-range observation.
quality_class = (ee.Image.constant(0).where(n_basic_range.gt(0),1).where(n_moderate.gt(0),2)
                 .where(n_primary.gt(0),3).rename('QUALITY_CLASS').clip(city))


## 6. Calculate municipal scene, base-QA and strict-QA area coverage

In [ ]:
# Calculate area using the same UTM grid/scale for every numerator and denominator.
pxarea = ee.Image.pixelArea().clip(city)
area_bands = ee.Image.cat([
    pxarea.rename('AREA_CITY_M2'),
    pxarea.updateMask(scene_footprint.gt(0)).rename('AREA_SCENE_FOOTPRINT_M2'),
    pxarea.updateMask(n_base.gt(0)).rename('AREA_BASE_QA_LST_M2'),
    pxarea.updateMask(n_primary.gt(0)).rename('AREA_PRIMARY_QA_LST_M2'),
    pxarea.updateMask(n_moderate.gt(0)).rename('AREA_MODERATE_QA_LST_M2'),
    pxarea.updateMask(n_basic_range.gt(0)).rename('AREA_BASIC_RANGE_LST_M2'),
])
area_dict = area_bands.reduceRegion(reducer=ee.Reducer.sum(), geometry=city,
                                   crsTransform=GRID_TRANSFORM, crs=GRID_CRS,
                                   maxPixels=1000000000, tileScale=4).getInfo()
city_area_m2 = float(area_dict['AREA_CITY_M2'])
assert city_area_m2>0

def fraction(key):
    area=float(area_dict.get(key) or 0)
    pct=100.0*area/city_area_m2
    assert 0<=pct<=100.5, (key,pct)
    return {'area_km2':round(area/1e6,4),'municipal_area_pct':round(pct,3)}

coverage_summary={
    'status':'SERVER_EE_CALCULATED_NOT_YET_RASTER_VERIFIED',
    'source_municipal_sha256': SOURCE_MUNICIPAL_SHA256,
    'municipal_grid_area_km2':round(city_area_m2/1e6,4),
    'municipal_planar_reference_km2':round(municipal_planar_km2,4),
    'scene_footprint':fraction('AREA_SCENE_FOOTPRINT_M2'),
    'base_qa_valid_lst':fraction('AREA_BASE_QA_LST_M2'),
    'strict_v05_primary_valid_lst':fraction('AREA_PRIMARY_QA_LST_M2'),
    'moderate_sensitivity':fraction('AREA_MODERATE_QA_LST_M2'),
    'basic_range_sensitivity':fraction('AREA_BASIC_RANGE_LST_M2'),
    'grid_transform':GRID_TRANSFORM,
    'original_strict_raster_drive_id':ORIGINAL_STRICT_RASTER_DRIVE_ID,
    'number_of_scenes':scene_count,
    'start_inclusive':START_DATE, 'end_exclusive':END_EXCLUSIVE,
    'percentage_denominator':'100 × covered 30m projected pixel area / all municipal projected pixel area',
    'temperature_unit':'deg C', 'area_unit':'km2',
    'intrinsic_thermal_resolution_note':'Landsat 8/9 TIRS nominal 100m; product grid resampled to 30m',
    'method':'Original V05 strict unchanged; moderate and basic-range separate exploratory masks',
}
print(json.dumps(coverage_summary,indent=2))
print('Old V05 20.57% was original study-grid valid-cell area / city, NOT the result of this new export.')
coverage_json=OUTPUT_DIR/f'{RUN_KEY}_AREA_COVERAGE_PRE_EXPORT.json'
coverage_json.write_text(json.dumps(coverage_summary,indent=2),encoding='utf-8')

## 7. Export the new diagnostic GeoTIFF

New V04 prefix; original V01 TIFF untouched. Repeated Run all recovers matching tasks, not duplicate exports.


In [ ]:
# Original four bands plus separate basic/moderate temperatures, counts, diagnostics and quality class.
# NoData is -9999; unknown coverage is NOT a 0 C measurement.
if not globals().get('AUTH_EE_READY', False):
    raise RuntimeError('First rerun Section 3 and pass its EE task-list preflight.')

export_image=(ee.Image.cat([city_median_lst,n_primary,n_base,scene_footprint,base_median,moderate_median,base_range_median,n_moderate,n_basic_range,base_qa_median,base_cdist_median,quality_class])
              .clip(city).unmask(NODATA,sameFootprint=False).toFloat())
task = None
if RUN_EXPORT:
    try:
        task_snapshot = check_earth_engine_batch_access()
    except Exception as exc:
        raise RuntimeError(
            'EE task listing failed; this cell did NOT submit any export. '
            'Rerun Section 3 with full OAuth scopes before retrying. '
            'Original error: '+str(exc)[:350]
        ) from exc

    matching=[(t,t.status()) for t in task_snapshot
              if t.status().get('description') == RUN_KEY]
    if matching:
        # Prevent accidental duplicated, overwritten, or ambiguously named exports.
        if len(matching)>1:
            raise RuntimeError(f'Multiple EE tasks already use {RUN_KEY}; inspect them manually.')
        task=matching[0][0]
        print('EXISTING TASK; NO NEW EXPORT:',task.id,
              'state:',matching[0][1].get('state'))
        print('Inspect Section 8. Never overwrite an archived export.')
    else:
        candidate=ee.batch.Export.image.toDrive(
            image=export_image, description=RUN_KEY, folder=OUTPUT_FOLDER_NAME,
            fileNamePrefix=RUN_KEY, region=ee.Geometry.Rectangle(GRID_BOUNDS, proj=GRID_CRS, geodesic=False),
            crsTransform=GRID_TRANSFORM,crs=GRID_CRS, maxPixels=10000000000,
            fileFormat='GeoTIFF',
            formatOptions={'cloudOptimized':True,'noData':NODATA})
        try:
            candidate.start()
        except Exception as exc:
            # The server may have accepted an export even if the response failed.
            raise RuntimeError(
                'Task submission failed or has ambiguous status. Inspect EE '
                'Tasks before retrying; do not bypass duplicate checks. '
                'Original error: '+str(exc)[:350]
            ) from exc
        task=candidate
        print('TASK SUBMITTED:',task.id,'state:',task.status().get('state'))
        print('Do not download until Section 8 confirms COMPLETED.')
else:
    print('RUN_EXPORT=False; no new export launched.')


## 8. Final run receipt (no waiting, no maps)

A successful `Run all` ends after submitting/recovering the export and reporting
its current status. Reopen Colab/Google Earth Engine Tasks to check for
`COMPLETED`. Then verify actual GeoTIFF bytes in the existing Drive output
folder before performing any map or pixel analysis in ChatGPT Python.

In [ ]:
if not globals().get('AUTH_EE_READY',False):
    raise RuntimeError('Section 3 batch-scope authorization is required.')
if task is None:
    saved=[t for t in check_earth_engine_batch_access()
           if t.status().get('description')==RUN_KEY]
    if len(saved)>1:
        raise RuntimeError('Ambiguous duplicate EE tasks; inspect before any export.')
    if saved:
        task=saved[0]
if task is None:
    run_status={'state':'NOT_SUBMITTED','task_id':None,'run_key':RUN_KEY}
else:
    raw_status=task.status()
    run_status={
        'state':raw_status.get('state'),
        'task_id':task.id,
        'run_key':RUN_KEY,
        'description':raw_status.get('description'),
        'error_message':raw_status.get('error_message'),
    }
run_status.update({
    'notebook_version':MAP003_WORKING_VERSION,
    'archive_event':MAP003_ARCHIVE_EVENT,
    'source_polygon_sha256':SOURCE_MUNICIPAL_SHA256,
    'output_drive_folder_id':OUTPUT_FOLDER_ID,
    'output_expected_prefix':RUN_KEY,
    'raster_status':'UNVERIFIED_UNTIL_COMPLETED_AND_DRIVE_READBACK',
    'scene_manifest_local_path':str(scene_csv),
    'coverage_metadata_local_path':str(coverage_json),
})
status_path=OUTPUT_DIR/f'{RUN_KEY}_TASK_STATUS.json'
status_path.write_text(json.dumps(run_status,indent=2,ensure_ascii=False),encoding='utf-8')
print('MAP003 V04 RUN-ALL RECEIPT')
print(json.dumps(run_status,indent=2,ensure_ascii=False))
print('DATA EXPORT: ASYNCHRONOUS; no figure/raster QA is performed in Colab.')
print('If status is COMPLETED, check the genuine GeoTIFF in this existing Drive folder:')
print('https://drive.google.com/drive/folders/'+OUTPUT_FOLDER_ID)
print('Local metadata:',scene_csv,coverage_json,status_path)

# Persist the small extraction provenance files without converting or plotting
# any raster. This is intentionally nonfatal: a Drive-auth problem must not
# disguise or undo a possibly accepted Earth Engine background export.
metadata_files = [municipality_file, scene_csv, coverage_json, status_path]
try:
    from google.colab import auth as colab_auth
    colab_auth.authenticate_user()
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload
    credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
    drive_service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
    for source_path in metadata_files:
        digest = hashlib.sha256(source_path.read_bytes()).hexdigest()
        # SHA-suffixed file names are immutable and repeated Run all is idempotent.
        export_name = f'{source_path.stem}__sha256_{digest[:12]}{source_path.suffix}'
        safe_name = export_name  # expected stable machine-generated names contain no apostrophes
        query = f"name = '{safe_name}' and '{OUTPUT_FOLDER_ID}' in parents and trashed = false"
        exists = drive_service.files().list(q=query, fields='files(id,name)',pageSize=10).execute().get('files',[])
        if exists:
            print('METADATA ALREADY IN DRIVE:', export_name, exists[0]['id'])
            continue
        mime = 'text/csv' if source_path.suffix.lower() == '.csv' else 'application/geo+json' if source_path.suffix.lower() == '.geojson' else 'application/json'
        media = MediaFileUpload(str(source_path), mimetype=mime, resumable=False)
        uploaded = drive_service.files().create(
            body={'name':export_name,'parents':[OUTPUT_FOLDER_ID]},
            media_body=media, fields='id,name').execute()
        print('METADATA SAVED TO DRIVE:', uploaded['name'], uploaded['id'])
except Exception as exc:
    print('WARNING: Small metadata files were not all confirmed in Google Drive:', str(exc)[:300])
    print('The Earth Engine task state above remains authoritative; inspect the task')
    print('and save the metadata files from', OUTPUT_DIR, 'before Colab runtime expires.')